<a href="https://colab.research.google.com/github/f1725developmenttechnologies-create/klarixa-ecosistema-ia/blob/main/Klarixa_ecosistema_ia_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import httpx
from pydantic import BaseModel, Field
from typing import Optional, Dict, Any

class AgentGroqConfig(BaseModel):
    agent_id: str
    groq_api_key: str = Field(default_factory=lambda: os.getenv("GROQ_API_KEY", ""))
    model_name: str = "gemma2-9b-it"

class BandGemmaAgent:
    def __init__(self, config: AgentGroqConfig):
        self.config = config
        self.client = httpx.AsyncClient(timeout=10.0)

    @classmethod
    async def create(cls, raw_config: dict) -> "BandGemmaAgent":
        config = AgentGroqConfig(**raw_config)
        if not config.groq_api_key:
            raise ValueError("Falta la GROQ_API_KEY para activar el cerebro de inferencia.")

        instance = cls(config)
        await instance._test_connection()
        return instance

    async def _test_connection(self):
        # Handshake rápido con la API de Groq
        pass

    async def query_brain(self, prompt: str) -> str:
        url = "https://api.groq.com/openai/v1/chat/completions"
        headers = {
            "Authorization": f"Bearer {self.config.groq_api_key}",
            "Content-Type": "application/json"
        }
        payload = {
            "model": self.config.model_name,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.2
        }
        resp = await self.client.post(url, headers=headers, json=payload)
        data = resp.json()
        return data["choices"][0]["message"]["content"]